# 00 — I due scraping insieme

Per avere più testate si uniscono:

- il **nuovo scraping** (ottobre 2026, `0-scraping`): `Dataset/NuovoScraping/dataset_articoli.csv`, già filtrato da
  `0-scraping/costruisci_dataset.ipynb`;
- il **vecchio scraping** di IrForGood (marzo 2025), **grezzo**: `Dataset/VecchioScraping/onlyPoliticsAutoScraping.csv`.

Al vecchio si applicano **gli stessi filtri** di `costruisci_dataset` (più di 100 parole; niente doppioni di URL o
di testo nella stessa testata; solo inglese, langdetect con seed 0 sulle prime 400 parole). Poi tutti e due
passano dalla stessa pulizia (`01`–`03`). Le etichette dei due file non si usano: in `03` si prendono per tutti
dallo stesso `newsguard.csv`.

Le date: il nuovo scraping ha la data della pagina (trafilatura); per il vecchio si ricava dall'indirizzo
dell'articolo quando la contiene (`/2024/05/…`), altrimenti resta vuota. `data_scraping` serve a `01` per tenere gli
articoli recenti rispetto a quando sono stati raccolti.

Le testate presenti in tutti e due gli scraping uniscono i loro articoli; i doppioni li toglie `01`.

**Sul server:** `cd ~/John/WebScience2027/1-data-cleaning && papermill -k mbvenv 00_unione_dataset.ipynb ../Results/1-data-cleaning/esecuzioni/00_unione_dataset.ipynb`

In [1]:
import ast
import os
import re

import pandas as pd
from langdetect import DetectorFactory, LangDetectException, detect

from comune import DATASET, DERIVED, NUOVO, VECCHIO

os.makedirs(DERIVED, exist_ok=True)

In [2]:
DATA_SCRAPING_NUOVO = "2026-10-13"      # avvio dello scraper (0-scraping/README.md)
DATA_SCRAPING_VECCHIO = "2025-03-01"    # il file grezzo del vecchio scraping è di marzo 2025
MIN_PAROLE = 100

## Il nuovo scraping

In [3]:
pulisci_dominio = lambda s: s.str.lower().str.strip().str.removeprefix("www.")
nuovo = pd.read_csv(NUOVO).rename(columns={"dominio": "domain", "testo": "text"})
nuovo = nuovo.assign(domain=pulisci_dominio(nuovo.domain), origine="nuovo (ottobre 2026)",
                     data=pd.to_datetime(nuovo.data, errors="coerce"), data_scraping=pd.Timestamp(DATA_SCRAPING_NUOVO))
print(f"nuovo scraping: {len(nuovo)} articoli, {nuovo.domain.nunique()} testate, con una data il {nuovo.data.notna().mean():.0%}")

nuovo scraping: 54532 articoli, 354 testate, con una data il 100%


## Il vecchio scraping, con gli stessi filtri

In [4]:
v = pd.read_csv(VECCHIO, low_memory=False)
v["text"] = v.text.fillna("")
v["domain"] = pulisci_dominio(v.domain)
passi = [("grezzo", len(v), v.domain.nunique())]
v = v[v.text.str.split().str.len() > MIN_PAROLE]
passi.append((f"più di {MIN_PAROLE} parole", len(v), v.domain.nunique()))
v = v.drop_duplicates(["domain", "url"]).drop_duplicates(["domain", "text"])
passi.append(("senza doppioni nella stessa testata", len(v), v.domain.nunique()))

DetectorFactory.seed = 0


def lingua(testo):
    try:
        return detect(testo)
    except LangDetectException:
        return "?"


v["lingua"] = v.text.str.split().str[:400].str.join(" ").map(lingua)
print(v.lingua.value_counts().head(5).to_string())
v = v[v.lingua == "en"]
passi.append(("solo inglese", len(v), v.domain.nunique()))
pd.DataFrame(passi, columns=["passo", "articoli", "testate"]).set_index("passo")

lingua
en    19369
de     5241
it     1580
fr      549
es      106


,articoli,testate
passo,,
grezzo,26909,527
più di 100 parole,26909,527
senza doppioni nella stessa testata,26909,527
solo inglese,19369,436


In [5]:
def data_da_url(u):
    m = re.search(r"/(20[0-2]\d)[/-](0?[1-9]|1[0-2])(?:[/-](0?[1-9]|[12]\d|3[01]))?(?=/|-|$)", str(u))
    return pd.Timestamp(int(m[1]), int(m[2]), int(m[3] or 1)) if m else pd.NaT


def primo_autore(a):
    if pd.isna(a):
        return None
    s = str(a).strip()
    if s.startswith("["):
        try:
            lista = ast.literal_eval(s)
            return lista[0] if lista else None
        except (ValueError, SyntaxError):
            pass
    return s


v = v.assign(origine="vecchio (IrForGood, marzo 2025)", titolo=v.title, autore=v.authors.map(primo_autore),
             data=v.url.map(data_da_url), data_scraping=pd.Timestamp(DATA_SCRAPING_VECCHIO))
print(f"vecchio scraping: {len(v)} articoli, {v.domain.nunique()} testate; data ricavata dall'indirizzo per il {v.data.notna().mean():.0%}")
print("anni (quando c'è la data):", v.data.dt.year.value_counts().sort_index().to_dict())

vecchio scraping: 19369 articoli, 436 testate; data ricavata dall'indirizzo per il 31%
anni (quando c'è la data): {2000.0: 1, 2001.0: 2, 2006.0: 1, 2007.0: 5, 2011.0: 2, 2012.0: 2, 2013.0: 13, 2014.0: 2, 2015.0: 17, 2016.0: 8, 2017.0: 38, 2018.0: 25, 2019.0: 36, 2020.0: 44, 2021.0: 50, 2022.0: 47, 2023.0: 174, 2024.0: 546, 2025.0: 4972}


## Insieme

In [6]:
colonne = ["domain", "origine", "url", "titolo", "data", "data_scraping", "autore", "text"]
uniti = pd.concat([nuovo[colonne], v[colonne]], ignore_index=True).rename_axis("article_id").reset_index()
dn, dv = set(nuovo.domain), set(v.domain)
print(f"testate: solo nel nuovo {len(dn - dv)}, solo nel vecchio {len(dv - dn)}, in tutti e due {len(dn & dv)}; totale {len(dn | dv)}")
print(f"articoli: {len(uniti)}")
uniti.to_csv(DATASET, index=False)
print("salvato", DATASET)

testate: solo nel nuovo 125, solo nel vecchio 207, in tutti e due 229; totale 561
articoli: 73901


salvato ../Dataset/derived/00_articoli_uniti.csv
